<a href="https://colab.research.google.com/github/themudasirpasha/redrob-ranking/blob/main/Ranking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install sentence-transformers scikit-learn pandas numpy python-docx

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json

candidates = []
with open('/content/drive/MyDrive/candidates.jsonl', 'r') as f:
    for line in f:
        if line.strip():
            candidates.append(json.loads(line))

print(f"Total: {len(candidates)}")

Total: 100000


In [52]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('BAAI/bge-base-en-v1.5')
print("BGE Model loaded!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

BGE Model loaded!


In [53]:
from datetime import datetime, date

consulting = ['tcs', 'infosys', 'wipro', 'accenture', 'cognizant', 'capgemini']

good_titles = [
    'ml engineer', 'machine learning engineer', 'ai engineer',
    'nlp engineer', 'senior ai engineer', 'applied ml',
    'data scientist', 'research engineer', 'applied scientist',
    'recommendation', 'search engineer', 'ranking engineer',
    'senior software engineer (ml)', 'staff machine learning',
    'lead ai', 'principal ml', 'deep learning engineer',
    'ai specialist', 'ai research', 'senior data scientist',
    'senior ml engineer', 'senior machine learning',
    'junior ml engineer', 'staff ml', 'lead ml'
]

filtered_candidates = []
for c in candidates:
    yoe = c['profile']['years_of_experience']
    country = c['profile']['country']
    sig = c['redrob_signals']
    title = c['profile']['current_title'].lower()

    # Basic filters
    if not (4 <= yoe <= 15): continue
    if country != 'India': continue
    if not sig['open_to_work_flag']: continue
    if sig['notice_period_days'] > 90: continue

    # Only good titles
    if not any(x in title for x in good_titles): continue

    # Consulting only
    all_consulting = all(
        any(x in job['company'].lower() for x in consulting)
        for job in c['career_history']
    )
    if all_consulting: continue

    # Activity filters
    last_active = datetime.strptime(sig['last_active_date'], '%Y-%m-%d').date()
    days_inactive = (date.today() - last_active).days
    if days_inactive > 180: continue

    # Profile quality
    if sig['profile_completeness_score'] < 30: continue
    if sig['recruiter_response_rate'] == 0: continue

    # Honeypot detection
    is_honeypot = False
    zero_month_experts = sum(1 for s in c['skills']
                            if s['proficiency'] == 'expert' and s.get('duration_months', 0) == 0)
    if zero_month_experts >= 3: is_honeypot = True
    if not sig['verified_email'] and not sig['verified_phone']: is_honeypot = True
    if sig['offer_acceptance_rate'] > 1: is_honeypot = True
    if sig['interview_completion_rate'] > 1: is_honeypot = True
    if is_honeypot: continue

    filtered_candidates.append(c)

print(f"Filtered: {len(filtered_candidates)}")

Filtered: 374


In [77]:
preferred = ['pune', 'noida', 'delhi', 'mumbai', 'hyderabad', 'bangalore']

filtered_candidates = [c for c in filtered_candidates
                       if any(loc in c['profile']['location'].lower() for loc in preferred)
                       or c['redrob_signals']['willing_to_relocate']]

print(f"Final filtered: {len(filtered_candidates)}")

Final filtered: 214


In [78]:
filtered_candidates = [c for c in filtered_candidates
                       if c['redrob_signals']['open_to_work_flag']
                       and (c['redrob_signals']['verified_email']
                            or c['redrob_signals']['verified_phone'])]

print(f"Final: {len(filtered_candidates)}")

Final: 214


In [ ]:
import numpy as np

def candidate_to_text(c):
    p = c['profile']
    skills = ', '.join([f"{s['name']} ({s['proficiency']})" for s in c['skills']])
    career = ' '.join([f"{j['title']} at {j['company']} ({j['duration_months']} months)"
                       for j in c['career_history']])
    return f"{p['current_title']} with {p['years_of_experience']} years. Location: {p['location']}, {p['country']}. Skills: {skills}. Career: {career}. {p['summary']}"

texts = [candidate_to_text(c) for c in filtered_candidates]
embeddings = model.encode(texts, batch_size=128, show_progress_bar=True)
np.save('/content/drive/MyDrive/embeddings_v2.npy', embeddings)
print(f"Done! Shape: {embeddings.shape}")

In [80]:
from docx import Document
from sklearn.metrics.pairwise import cosine_similarity

doc = Document('/content/drive/MyDrive/job_description.docx')
jd_text = '\n'.join([para.text for para in doc.paragraphs])
jd_embedding = model.encode([jd_text])
semantic_scores = cosine_similarity(jd_embedding, embeddings)[0]
print("Semantic done! Max:", semantic_scores.max())

Semantic done! Max: 0.86832577


In [81]:
def skills_score(c):
    must_have = [
        'embeddings', 'faiss', 'pinecone', 'weaviate', 'qdrant', 'milvus',
        'opensearch', 'elasticsearch', 'vector', 'sentence-transformers',
        'python', 'ranking', 'retrieval', 'nlp', 'machine learning',
        'deep learning', 'ndcg', 'mrr', 'map', 'search', 'recommendation',
        'llm', 'transformers', 'pytorch', 'tensorflow', 'fine-tuning',
        'rag', 'hybrid search', 'bm25', 'reranking'
    ]

    nice_to_have = [
        'lora', 'qlora', 'peft', 'xgboost', 'learning to rank',
        'distributed', 'inference', 'a/b testing'
    ]

    # NEW — endorsement weighting
    must_matched = 0
    for skill in must_have:
        for s in c['skills']:
            if skill in s['name'].lower():
                endorsements = s.get('endorsements', 0)
                weight = 1.0 + min(endorsements / 50, 0.5)  # max 1.5x boost
                must_matched += weight
                break

    must_score = min(must_matched / len(must_have), 1.0)

    candidate_skills = [s['name'].lower() for s in c['skills']]

    nice_matched = sum(1 for skill in nice_to_have
                      if any(skill in cs for cs in candidate_skills))
    nice_score = (nice_matched / len(nice_to_have)) * 0.2

    assessments = c['redrob_signals'].get('skill_assessment_scores', {})
    assessment_bonus = 0
    if assessments:
        avg = sum(assessments.values()) / len(assessments)
        assessment_bonus = (avg / 100) * 0.1

    return min(must_score + nice_score + assessment_bonus, 1.0)

skills_scores = [skills_score(c) for c in filtered_candidates]
print("Skills scoring done! Max:", max(skills_scores))

Skills scoring done! Max: 0.6951533333333334


In [67]:
def career_depth_score(c):
    score = 0

    # Production keywords — JD ne kaha "shipped to real users"
    production_keywords = [
        'production', 'deployed', 'shipped', 'real users', 'scale',
        'latency', 'serving', 'inference', 'ranking system', 'search system',
        'recommendation system', 'retrieval system', 'embedding', 'vector'
    ]

    # Pure researcher penalty — JD ne explicitly kaha nahi chahiye
    research_only_keywords = ['paper', 'arxiv', 'academic', 'thesis',
                              'research lab', 'publication']

    # Job hopper check — JD ne kaha "switching every 1.5 years"
    short_tenures = sum(1 for job in c['career_history']
                       if job['duration_months'] < 18)
    total_jobs = len(c['career_history'])

    ai_ml_titles = ['ml', 'ai', 'machine learning', 'nlp', 'search',
                    'recommendation', 'scientist', 'ranking', 'applied',
                    'deep learning', 'data scientist']

    # 1. Production experience check
    for job in c['career_history']:
        desc = job.get('description', '').lower()
        if any(kw in desc for kw in production_keywords):
            score += 0.3
            break

    # 2. AI/ML title months
    total_ai_months = sum(job['duration_months'] for job in c['career_history']
                         if any(x in job['title'].lower() for x in ai_ml_titles))
    if total_ai_months >= 48: score += 0.3
    elif total_ai_months >= 24: score += 0.2
    elif total_ai_months >= 12: score += 0.1

    # 3. Job hopper penalty
    if total_jobs > 0:
        hopper_ratio = short_tenures / total_jobs
        if hopper_ratio > 0.7: score -= 0.2
        elif hopper_ratio > 0.5: score -= 0.1

    # 4. Pure researcher penalty
    research_count = 0
    for job in c['career_history']:
        desc = job.get('description', '').lower()
        if any(kw in desc for kw in research_only_keywords):
            research_count += 1
    if research_count == len(c['career_history']): score -= 0.3

    # 5. Product company bonus
    consulting = ['tcs', 'infosys', 'wipro', 'accenture', 'cognizant', 'capgemini']
    for job in c['career_history']:
        if not any(x in job['company'].lower() for x in consulting):
            if job['duration_months'] >= 12:
                score += 0.2
                break

    # 6. GitHub activity — open source contributions
    github = c['redrob_signals'].get('github_activity_score', -1)
    if github > 50: score += 0.2
    elif github > 20: score += 0.1

    return min(max(score, 0), 1.0)

career_scores = [career_depth_score(c) for c in filtered_candidates]
print("Career depth done! Max:", max(career_scores))
print("Min:", min(career_scores))

Career depth done! Max: 1.0
Min: 0.5


In [69]:
def experience_score(c):
    score = 0
    yoe = c['profile']['years_of_experience']
    sig = c['redrob_signals']

    # JD ne kaha "6-8 years ideal, 4-5 ok, outside band if signals strong"
    if 6 <= yoe <= 8: score += 0.5      # Perfect
    elif 5 <= yoe <= 9: score += 0.4    # Good
    elif 4 <= yoe <= 12: score += 0.2   # Acceptable
    else: score += 0.1                   # Outside band

    # Location — JD ne kaha Pune/Noida preferred
    top_locations = ['pune', 'noida']
    good_locations = ['delhi', 'mumbai', 'hyderabad', 'bangalore']
    location = c['profile']['location'].lower()

    if any(loc in location for loc in top_locations): score += 0.3
    elif any(loc in location for loc in good_locations): score += 0.2
    elif sig['willing_to_relocate']: score += 0.1

    # Education tier bonus
    for edu in c.get('education', []):
        if edu.get('tier') == 'tier_1':
            score += 0.2
            break
        elif edu.get('tier') == 'tier_2':
            score += 0.1
            break

    return min(score, 1.0)

experience_scores = [experience_score(c) for c in filtered_candidates]
print("Experience scoring done! Max:", max(experience_scores))
print("Min:", min(experience_scores))

Experience scoring done! Max: 1.0
Min: 0.30000000000000004


In [71]:
def availability_score(c):
    sig = c['redrob_signals']
    score = 0

    from datetime import datetime, date
    last_active = datetime.strptime(sig['last_active_date'], '%Y-%m-%d').date()
    days_inactive = (date.today() - last_active).days
    if days_inactive <= 14: score += 0.20
    elif days_inactive <= 30: score += 0.15
    elif days_inactive <= 60: score += 0.08
    elif days_inactive <= 90: score += 0.04

    score += sig['recruiter_response_rate'] * 0.15

    if sig['notice_period_days'] <= 30: score += 0.15
    elif sig['notice_period_days'] <= 60: score += 0.08
    elif sig['notice_period_days'] <= 90: score += 0.04

    saved = sig.get('saved_by_recruiters_30d', 0)
    if saved >= 5: score += 0.15
    elif saved >= 3: score += 0.10
    elif saved >= 1: score += 0.05

    github = sig.get('github_activity_score', -1)
    if github > 70: score += 0.10
    elif github > 40: score += 0.07
    elif github > 20: score += 0.03

    score += sig.get('interview_completion_rate', 0) * 0.08

    oar = sig.get('offer_acceptance_rate', -1)
    if oar > 0: score += oar * 0.05

    if sig.get('linkedin_connected'): score += 0.05

    score += (sig['profile_completeness_score'] / 100) * 0.05

    endorsements = sig.get('endorsements_received', 0)
    if endorsements >= 20: score += 0.02
    elif endorsements >= 10: score += 0.01

    search = sig.get('search_appearance_30d', 0)
    if search >= 100: score += 0.05
    elif search >= 50: score += 0.03
    elif search >= 10: score += 0.01

    # NEW — active job seeker signal
    apps = sig.get('applications_submitted_30d', 0)
    if apps >= 5: score += 0.05
    elif apps >= 3: score += 0.03
    elif apps >= 1: score += 0.01

    # NEW — platform engagement
    connections = sig.get('connection_count', 0)
    if connections >= 100: score += 0.04
    elif connections >= 50: score += 0.02
    elif connections >= 20: score += 0.01

    return min(score, 1.0)

availability_scores = [availability_score(c) for c in filtered_candidates]
print("Availability scoring done! Max:", max(availability_scores))
print("Min:", min(availability_scores))

Availability scoring done! Max: 0.9320000000000002
Min: 0.49705


In [72]:
import numpy as np

skills_arr = np.array(skills_scores)
career_arr = np.array(career_scores)
experience_arr = np.array(experience_scores)
availability_arr = np.array(availability_scores)

# JD based weights
final_scores = (
    0.35 * semantic_scores +
    0.25 * skills_arr +
    0.22 * career_arr +
    0.10 * experience_arr +
    0.08 * availability_arr
)

print("Final scoring done!")
print("Max:", final_scores.max())
print("Min:", final_scores.min())

Final scoring done!
Max: 0.8281964304701488
Min: 0.5235828487167359


In [73]:
top_100_indices = np.argsort(final_scores)[::-1][:100]
print("Top 100 ready!")

Top 100 ready!


In [74]:
import pandas as pd
from datetime import datetime, date

def build_reasoning(c):
    p = c['profile']
    sig = c['redrob_signals']

    ai_roles = [j['title'] for j in c['career_history']
                if any(x in j['title'].lower() for x in
                       ['ml', 'ai', 'machine learning', 'nlp', 'search',
                        'recommendation', 'scientist', 'ranking', 'applied'])]

    key_skills = [s['name'] for s in c['skills']
                  if any(x in s['name'].lower() for x in
                         ['embedding', 'vector', 'faiss', 'pinecone', 'retrieval',
                          'ranking', 'llm', 'transformer', 'search', 'nlp',
                          'qdrant', 'weaviate', 'pytorch'])][:3]

    last_active = (date.today() - datetime.strptime(sig['last_active_date'], '%Y-%m-%d').date()).days

    role_str = f"AI/ML roles: {', '.join(ai_roles[:2])}" if ai_roles else "Applied ML background"
    skill_str = f"Core skills: {', '.join(key_skills)}" if key_skills else "General ML skills"
    avail_str = f"Active {last_active}d ago, {int(sig['recruiter_response_rate']*100)}% response rate, {sig['notice_period_days']}d notice"

    return f"{role_str}. {skill_str}. {avail_str}."

results = []
for rank, idx in enumerate(top_100_indices, 1):
    c = filtered_candidates[idx]
    results.append({
        'candidate_id': c['candidate_id'],
        'rank': rank,
        'score': round(final_scores[idx], 4),
        'reasoning': build_reasoning(c)
    })

df = pd.DataFrame(results)
df.to_csv('/content/drive/MyDrive/team_perfectmatchmakers.csv', index=False)
print("CSV saved!")
print(df.head(5))

CSV saved!
   candidate_id  rank   score  \
0  CAND_0018499     1  0.8282   
1  CAND_0077337     2  0.8114   
2  CAND_0086022     3  0.8102   
3  CAND_0081846     4  0.8010   
4  CAND_0007009     5  0.7767   

                                           reasoning  
0  AI/ML roles: Senior Machine Learning Engineer,...  
1  AI/ML roles: Staff Machine Learning Engineer, ...  
2  AI/ML roles: Senior Applied Scientist, Senior ...  
3  AI/ML roles: Lead AI Engineer, Senior Machine ...  
4  AI/ML roles: Recommendation Systems Engineer, ...  


In [75]:
print("Total rows:", len(df))
print("Columns:", list(df.columns))
print("Min rank:", df['rank'].min())
print("Max rank:", df['rank'].max())
print("Duplicate IDs:", df['candidate_id'].duplicated().sum())
print("Scores decreasing:", (df['score'].diff().dropna() <= 0).all())

Total rows: 100
Columns: ['candidate_id', 'rank', 'score', 'reasoning']
Min rank: 1
Max rank: 100
Duplicate IDs: 0
Scores decreasing: True
